# Lab W7: Adopsi Repo Eksternal

Kamu meng-clone repo eksternal (`huggingface/transformers`), fokus ke reference implementation `examples/pytorch/text-classification/`, menulis `repo_map.md` enam bagian, menjalankan smoke test 3 level pada kodenya, lalu mem-port satu komponen secara minimal-invasif (opsi focal loss untuk teks pincang kelas) dan membandingkannya dengan baseline pada SmSA. Pilihan repo ini mengikuti rekomendasi domain teks di bab W7 §D3; pola modifikasi minimal mengikuti §3.4. Konsep (§) merujuk ke `07_W7_Text_Transformers_Repo_Adoption.md` §3 dan §D1-D7 (Pendalaman Repo Adoption).

**Prasyarat:** Bab W7 §3 (urutan baca, repo_map.md, modifikasi minimal-invasif) dan §D2-D3 (urutan luar-ke-dalam, smoke test 3-level, navigasi grep, worked example) sudah dibaca, Lab utama W7 (text classification) selesai, familiar dengan `git clone/status/diff`, `grep`/`rg`, dan HuggingFace `transformers`. **Hardware & waktu:** CPU cukup untuk smoke test dan eksperimen ringan, GPU opsional, ~500 MB-1 GB storage (clone HF transformers + model kecil), ~3-5 jam. Butuh koneksi internet.

## Alur Lab

1. **Clone dan baca struktur:** temukan entry point, dependency, dan alur data di reference text-classification HF.
2. **Repo map:** tulis peta singkat sebelum mengubah kode.
3. **Smoke test bertingkat:** import test, forward dummy pada model teks kecil, lalu satu iterasi training.
4. **Satu port kecil:** tambahkan opsi focal loss untuk klasifikasi teks pincang kelas.
5. **Eksperimen kecil:** bandingkan CrossEntropy vs focal loss pada SmSA dengan seed terbatas.
6. **Git workflow:** pisahkan perubahan menjadi commit yang bisa direview.

## 0. Setup

Pasang `transformers` (umumnya sudah ada di Colab) dan kumpulkan import. `sklearn` dipakai untuk macro-F1 dan F1 per kelas.

In [ ]:
!pip install -q transformers

import os
import subprocess
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Clone Repo Eksternal

Bab W7 §D3 menyarankan `huggingface/transformers` untuk Capstone domain teks, dengan `examples/pytorch/text-classification/run_classification.py` sebagai skeleton yang paling mudah diadaptasi. Clone `--depth 1` (riwayat dangkal) supaya unduhan lebih ringan; tetap perlu 1-2 menit.

In [ ]:
_root = os.path.abspath("..")
external_dir = os.path.join(_root, "external")
os.makedirs(external_dir, exist_ok=True)

tf_dir = os.path.join(external_dir, "transformers")
if not os.path.exists(tf_dir):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/huggingface/transformers", tf_dir],
        check=True,
    )
    print("Repo cloned.")
else:
    print("Repo sudah ada, lewati clone.")

## 2. Eksplorasi Struktur: 15-Minute Map

Jelajahi `examples/pytorch/text-classification/` - reference implementation untuk klasifikasi teks di HuggingFace. Catat file utama dan perannya.

In [ ]:
ref_dir = os.path.join(tf_dir, 'examples', 'pytorch', 'text-classification')

print('=== examples/pytorch/text-classification/ ===')
for f in sorted(os.listdir(ref_dir)):
    fpath = os.path.join(ref_dir, f)
    size = os.path.getsize(fpath) if os.path.isfile(fpath) else '-'
    print(f'  {f:32s} ({size} bytes)' if isinstance(size, int) else f'  {f:32s}/')

print()
print('=== File kunci dan perannya ===')
key_files = {
    'run_classification.py': 'Entry point generik - argparse via HfArgumentParser, main(), Trainer. Skeleton paling mudah diadaptasi.',
    'run_glue.py': 'Varian untuk benchmark GLUE. Pola mirip run_classification.py.',
    'README.md': 'Cara menjalankan tiap script, argumen utama, contoh dataset.',
    'requirements.txt': 'Dependency: transformers, datasets, evaluate, scikit-learn, dll.',
}
for fname, desc in key_files.items():
    fpath = os.path.join(ref_dir, fname)
    if os.path.exists(fpath):
        lines = len(open(fpath, encoding='utf-8').readlines())
        print(f'  {fname:24s} ({lines:4d} lines) - {desc}')
    else:
        print(f'  {fname:24s} (MISSING) - {desc}')

In [ ]:
# Quick grep: temukan entry point dan cara metrik dihitung
target = os.path.join(ref_dir, 'run_classification.py')
print('=== Mencari def main / compute_metrics di run_classification.py ===')
with open(target, encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        s = line.strip()
        if s.startswith('def main') or 'def compute_metrics' in s or 'DataCollatorWithPadding' in s:
            print(f'  Line {i:4d}: {s[:90]}')

## 3. Tulis `repo_map.md`

Isi template 6-bagian di bawah. Ini adalah produk utama lab ini. Jangan copy-paste dari komentar kode - tulis dengan kata-kata sendiri setelah benar-benar membaca file yang dimaksud.

> [!TIP]
> Jangan tergoda melewati bagian `repo_map.md` karena "sudah paham". Pemetaan tertulis adalah produk utama lab ini, bukan kode yang di-port.

In [ ]:
repo_map_content = """
# repo_map.md - huggingface/transformers (examples/pytorch/text-classification/)

## 1. Tujuan Repo
[Tulis: apa yang dilakukan example ini? Klasifikasi teks dengan model HuggingFace. Apa bedanya dari training loop sederhana di template?]

## 2. Struktur Key Directory
[Tulis: isi examples/pytorch/text-classification/ dengan anotasi pendek per file]

## 3. Entry Point & Alur Eksekusi
[Tulis: dari `python run_classification.py --model_name_or_path ... --train_file ...`, apa yang terjadi? Bagaimana HfArgumentParser memproses argumen, model & tokenizer dimuat, dataset ditokenisasi, Trainer dijalankan?]

## 4. Cara Model Didefinisikan
[Tulis: dari mana model diambil? AutoModelForSequenceClassification. Bagaimana num_labels diset? Bagaimana head klasifikasi ditambahkan di atas backbone?]

## 5. Cara Data Masuk
[Tulis: dataset apa? Bagaimana tokenizer dipakai? Apa peran DataCollatorWithPadding (dynamic padding)? Bagaimana label diproses?]

## 6. Config & Hyperparameter
[Tulis: bagaimana hyperparameter diset? TrainingArguments. Parameter kunci: learning_rate, per_device_train_batch_size, num_train_epochs, dll.]
"""

map_path = os.path.join(_root, 'docs', 'repo_map_transformers.md')
os.makedirs(os.path.dirname(map_path), exist_ok=True)
with open(map_path, 'w', encoding='utf-8') as f:
    f.write(repo_map_content)
print(f'repo_map.md template ditulis ke {map_path}')
print('Isi setiap section [Tulis: ...] dengan observasi Anda sendiri.')

## 4. Smoke Test 3-Level pada Kode Eksternal

Verifikasi pipeline klasifikasi teks HuggingFace berfungsi sebelum kita memodifikasinya. Untuk smoke test cukup model kecil `prajjwal1/bert-tiny` (head klasifikasi berinisialisasi acak; yang diuji hanya jalannya pipeline, bukan akurasi).

In [ ]:
# Level 1: Import - apakah library tersedia?
print('=== Smoke Test Level 1: Import ===')
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
print(f'transformers version: {transformers.__version__}')

# Level 2: Forward dummy - apakah shape logits cocok dengan jumlah kelas?
print('\n=== Smoke Test Level 2: Forward Dummy ===')
smoke_name = 'prajjwal1/bert-tiny'
tok = AutoTokenizer.from_pretrained(smoke_name)
mdl = AutoModelForSequenceClassification.from_pretrained(smoke_name, num_labels=3)
enc = tok(['produk ini sangat bagus', 'pelayanan sangat mengecewakan'],
          padding=True, truncation=True, return_tensors='pt')
with torch.no_grad():
    out = mdl(**enc)
print(f'  Input ids: {tuple(enc["input_ids"].shape)}')
print(f'  Output logits: {tuple(out.logits.shape)}  (expected: (2, 3))')
assert out.logits.shape == (2, 3), f'Shape mismatch: {out.logits.shape}'
print('  Level 2 passed')

# Level 3: Satu iterasi training - apakah loss bisa dihitung dan gradient mengalir?
print('\n=== Smoke Test Level 3: Satu Iterasi Training ===')
mdl.train()
opt = torch.optim.AdamW(mdl.parameters(), lr=1e-4)
labels = torch.tensor([1, 0])
opt.zero_grad()
out = mdl(**enc, labels=labels)   # model menghitung CrossEntropy internal
out.loss.backward()
opt.step()
print(f'  Loss: {out.loss.item():.4f}')
print('  Level 3 passed - gradient mengalir, optimizer step berhasil')

## 5. Port Satu Komponen: Tambahkan Opsi Focal Loss untuk Teks

Mengikuti contoh worked example bab (§D3.4), kita menambah satu fitur seminimal mungkin: opsi focal loss. SmSA pincang kelasnya (kelas netral jarang), jadi focal loss yang menekan kontribusi sampel mudah berpotensi menaikkan F1 kelas minoritas. Kita pakai pola modifikasi minimal §3.4: `build_loss` dengan default `ce` (perilaku lama tetap), focal sebagai opsi, ditulis ke file baru `src/losses_text.py`.

In [ ]:
class FocalLoss(nn.Module):
    """Multi-class focal loss; gamma=0 identik dengan CrossEntropyLoss."""
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        if self.gamma == 0.0:
            return ce.mean()
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def build_loss(cfg):
    """Default 'ce' mempertahankan perilaku lama; 'focal' adalah opsi tambahan."""
    name = cfg.get('name', 'ce')
    if name == 'ce':
        return nn.CrossEntropyLoss()
    if name == 'focal':
        return FocalLoss(gamma=cfg.get('gamma', 2.0))
    raise ValueError(f'Unknown loss: {name}')


# Smoke test komponen baru: gamma=0 harus identik dengan CrossEntropy
logits = torch.randn(8, 3)
targets = torch.randint(0, 3, (8,))
ce_val = nn.CrossEntropyLoss()(logits, targets).item()
focal0 = FocalLoss(gamma=0.0)(logits, targets).item()
assert abs(ce_val - focal0) < 1e-5, 'FocalLoss(gamma=0) harus == CrossEntropy'
print(f'Smoke test port: CE={ce_val:.4f}, Focal(gamma=0)={focal0:.4f} -> identik. OK')
print(f'Focal(gamma=2) pada batch dummy: {FocalLoss(2.0)(logits, targets).item():.4f}')

In [ ]:
# Tulis ke file baru src/losses_text.py (modifikasi minimal: file baru, tidak mengubah default lama)
losses_path = os.path.join(_root, 'src', 'losses_text.py')
os.makedirs(os.path.dirname(losses_path), exist_ok=True)
with open(losses_path, 'w', encoding='utf-8') as f:
    f.write('''"""Loss untuk klasifikasi teks - opsi focal loss untuk kelas pincang.

Diadopsi setelah membaca examples/pytorch/text-classification/ di huggingface/transformers
(reference text-classification). Focal Loss: Lin et al., 2017 (arXiv:1708.02002).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class FocalLoss(nn.Module):
    """Multi-class focal loss; gamma=0 identik dengan CrossEntropyLoss."""

    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        if self.gamma == 0.0:
            return ce.mean()
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def build_loss(cfg: dict):
    name = cfg.get("name", "ce")
    if name == "ce":
        return nn.CrossEntropyLoss()
    if name == "focal":
        return FocalLoss(gamma=cfg.get("gamma", 2.0))
    raise ValueError(f"Unknown loss: {name}")
''')
print(f'Port selesai: {losses_path}')

## 6. Baseline (CE) vs Focal pada SmSA (2 Kondisi × 2 Seeds)

Bandingkan CrossEntropy vs focal loss pada classifier teks ringan. Kita lihat macro-F1 dan F1 kelas netral (kelas minoritas), karena di sanalah focal loss diharapkan paling terasa efeknya.

In [ ]:
# Muat SmSA + classifier teks ringan (self-contained, tidak mengimpor dari src/)
base_url = 'https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/'
labels = ['negatif', 'positif', 'netral']
label_map = {'positive': 1, 'positif': 1, 'neutral': 2, 'netral': 2, 'negative': 0, 'negatif': 0}
numeric_map = {'0': 1, '1': 2, '2': 0}

def load_smsa(split):
    fname = 'valid_preprocess.tsv' if split == 'valid' else f'{split}_preprocess.tsv'
    df = pd.read_csv(base_url + fname, sep='\t', names=['text', 'label'])
    raw = df['label'].astype(str).str.strip()
    df['label'] = raw.map(lambda x: label_map.get(x, numeric_map.get(x))).astype(int)
    return df['text'].tolist(), df['label'].tolist()

train_texts, train_labels = load_smsa('train')
val_texts, val_labels = load_smsa('valid')
dist = np.bincount(train_labels, minlength=3)
print('Distribusi kelas train:', dict(zip(labels, dist)), '(netral = minoritas)')

PAD, UNK, MAX_LEN = 0, 1, 64
counter = Counter(tok for t in train_texts for tok in t.lower().split())
vocab = {'<pad>': PAD, '<unk>': UNK}
for w, _ in counter.most_common(20000):
    vocab[w] = len(vocab)

def encode_text(text):
    ids = [vocab.get(t, UNK) for t in text.lower().split()][:MAX_LEN]
    return ids + [PAD] * (MAX_LEN - len(ids))

def make_dataset(texts, labs):
    X = torch.tensor([encode_text(t) for t in texts], dtype=torch.long)
    y = torch.tensor(labs, dtype=torch.long)
    return TensorDataset(X, y)

train_ds = make_dataset(train_texts, train_labels)
val_ds = make_dataset(val_texts, val_labels)

class TextMeanClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden=128, num_classes=3, pad_idx=PAD):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.head = nn.Sequential(nn.Linear(embed_dim, hidden), nn.ReLU(),
                                  nn.Dropout(0.3), nn.Linear(hidden, num_classes))
        self.pad_idx = pad_idx

    def forward(self, input_ids):
        mask = (input_ids != self.pad_idx).float().unsqueeze(-1)
        emb = self.embed(input_ids)
        pooled = (emb * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.head(pooled)

def set_seed(seed):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

print('Data + model siap. Vocab:', len(vocab))

In [ ]:
def train_eval(seed, loss_cfg, epochs=15, lr=1e-3):
    set_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=128)
    model = TextMeanClassifier(len(vocab)).to(device)
    crit = build_loss(loss_cfg)            # <- komponen yang baru di-port
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    model.eval()
    preds, golds = [], []
    with torch.no_grad():
        for x, y in val_loader:
            preds.extend(model(x.to(device)).argmax(1).cpu().numpy())
            golds.extend(y.numpy())
    macro = f1_score(golds, preds, average='macro')
    per_class = f1_score(golds, preds, average=None, labels=[0, 1, 2])
    return macro, per_class


conditions = {'CE': {'name': 'ce'}, 'Focal(g=2)': {'name': 'focal', 'gamma': 2.0}}
seeds = [42, 1337]
rows = []
for cond, cfg in conditions.items():
    for seed in seeds:
        t0 = time.time()
        macro, per_class = train_eval(seed, cfg)
        rows.append({'Kondisi': cond, 'Seed': seed, 'Macro-F1': macro,
                     'F1-netral': per_class[2]})
        print(f"{cond:10s} seed={seed} macro-F1={macro:.4f} F1-netral={per_class[2]:.4f} ({time.time()-t0:.0f}s)")

res = pd.DataFrame(rows)
print('\n=== Ringkasan (mean per kondisi) ===')
print(res.groupby('Kondisi')[['Macro-F1', 'F1-netral']].mean().to_string(float_format='{:.4f}'.format))

## 7. Git Workflow: 4+ Commits Terpisah

Commit history kecil dan bermakna. Jalankan dari terminal di folder `template/`:

```bash
# Commit 1: Struktur direktori
mkdir -p external docs
echo 'external/' >> .gitignore
git add .gitignore
git commit -m "chore: add external/ to .gitignore"

# Commit 2: Clone repo eksternal
git clone --depth 1 https://github.com/huggingface/transformers external/transformers
# (external/ tidak di-track - hanya lokal)

# Commit 3: repo_map.md
git add docs/repo_map_transformers.md
git commit -m "docs: add repo_map.md for HF text-classification reference"

# Commit 4: Port opsi focal loss ke src/
git add src/losses_text.py
git commit -m "feat: add focal loss option for imbalanced text classification"

# Commit 5 (opsional): Hasil perbandingan CE vs focal
git add experiments/
git commit -m "exp: add CE vs focal comparison on SmSA (2 seeds)"
```

Pastikan setiap commit punya scope yang jelas dan pesan informatif.

## 8. Draft PR Description

Tulis deskripsi Pull Request jika komponen yang di-port akan di-submit ke upstream `template`. Template:

### Draft PR: Add Focal Loss Option for Text Classification

**Summary:**

> *[Tulis 2-3 kalimat: apa yang ditambahkan, dari mana inspirasinya (reference text-classification HF)]*

**Changes:**
- `src/losses_text.py` (new): `FocalLoss` + `build_loss(cfg)`; default `ce` mempertahankan perilaku lama.

**Motivation:**

> *[Tulis: kenapa focal loss berguna untuk SmSA? Hubungkan ke ketidakseimbangan kelas netral dan bukti dari eksperimen.]*

**Testing:**
- Smoke test: `FocalLoss(gamma=0)` == `CrossEntropyLoss` pada batch dummy.
- Baseline comparison: SmSA TextMeanClassifier, CE vs focal, 2 seeds per kondisi.

**Results (if available):**

| Condition | Seed | Macro-F1 | F1-netral |
| --- | --- | --- | --- |
| CE | 42 | ... | ... |
| CE | 1337 | ... | ... |
| Focal | 42 | ... | ... |
| Focal | 1337 | ... | ... |

## 9. Refleksi

Jawab di sel di bawah:

1. **Berapa lama total: clone → map → port?** Apakah < 3 jam seperti target §D3 W7?

2. **Apa satu pola kode dari `huggingface/transformers` yang Anda adopsi** ke gaya penulisan sendiri? (contoh: pemisahan argumen via dataclass, struktur `build_*`, pola `compute_metrics`, dll.)

3. **Apa bagian yang TIDAK Anda port**, meskipun menarik - kenapa? (scope, dependency, kompleksitas)

4. **Apa yang paling tidak terduga** tentang struktur kode di `huggingface/transformers` dibanding template? (mis. ukuran repo, peran `Trainer`, banyaknya abstraksi)

### Jawaban Refleksi

**1. Durasi total:**

> *[tulis di sini]*

**2. Pola kode yang diadopsi:**

> *[tulis di sini]*

**3. Bagian yang tidak di-port dan alasannya:**

> *[tulis di sini]*

**4. Hal paling tidak terduga:**

> *[tulis di sini]*

## Self-Check Quick

- [ ] Repo eksternal (`huggingface/transformers`) di-clone.
- [ ] `repo_map.md` enam bagian (tujuan, struktur, entry point, model, data, config) ditulis untuk reference text-classification.
- [ ] Smoke test 3 level pada kode eksternal: L1 import, L2 forward dummy pada model teks, L3 satu iterasi training.
- [ ] 1 komponen di-port minimal-invasif ke `template/src/` (file baru `losses_text.py`, default lama tidak berubah).
- [ ] Baseline + variasi dijalankan (CE vs focal, 2 kondisi x 2 seed) pada SmSA dengan macro-F1 dan F1 per kelas.
- [ ] Commit history kecil dan bermakna (4+ commit terpisah).